# Lab 4 — Python i bazy danych


In [ ]:
import sys
!{sys.executable} -m pip install requests pymongo -q


### Zadanie 1 · PostgreSQL (lub SQLite)

Korzystając z **Random User API** (`https://randomuser.me/api/?results=30`), wygeneruj 30 użytkowników i zapisz ich dane (imię, nazwisko, email, adres, wiek, płeć) w tabeli SQL.

Następnie sprawdź SQL-em:
- Ile jest mężczyzn, a ile kobiet?
- Jaki jest średni wiek?
- W ilu krajach mieszkają?

*Tip: `requests.get(...).json()["results"]` zwraca listę dictów — iteruj i INSERT-uj po jednym.*  
*Możesz użyć SQLite (jak w tym notebooku) lub PostgreSQL jeśli masz zainstalowany.*


In [ ]:
import sqlite3
import requests

API_URL = "https://randomuser.me/api/?results=30"


def dane_przykladowe():
    kraje = ["Poland", "Germany", "France", "Spain", "Italy"]
    users = []

    for i in range(30):
        gender = "male" if i % 2 == 0 else "female"
        users.append({
            "gender": gender,
            "name": {
                "first": f"Imie{i + 1}",
                "last": f"Nazwisko{i + 1}",
            },
            "email": f"user{i + 1}@example.com",
            "location": {
                "street": {
                    "number": i + 1,
                    "name": "Testowa",
                },
                "city": "Krakow",
                "country": kraje[i % len(kraje)],
            },
            "dob": {
                "age": 20 + (i % 40),
            },
        })

    return users


try:
    response = requests.get(API_URL, timeout=10)
    response.raise_for_status()
    users = response.json()["results"]
except Exception:
    users = dane_przykladowe()

print(f"Liczba użytkowników: {len(users)}")

with sqlite3.connect("lab4_users.sqlite") as conn:
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS Users")

    cur.execute("""
        CREATE TABLE Users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name TEXT,
            last_name TEXT,
            email TEXT,
            address TEXT,
            age INTEGER,
            gender TEXT,
            country TEXT
        )
    """)

    for user in users:
        first_name = user["name"]["first"]
        last_name = user["name"]["last"]
        email = user["email"]
        gender = user["gender"]
        age = user["dob"]["age"]
        country = user["location"]["country"]
        street = user["location"]["street"]
        address = f'{street["number"]} {street["name"]}, {user["location"]["city"]}, {country}'

        cur.execute(
            """
            INSERT INTO Users (first_name, last_name, email, address, age, gender, country)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            (first_name, last_name, email, address, age, gender, country),
        )

    conn.commit()

    print("Liczba osób według płci:")
    cur.execute("""
        SELECT
            CASE gender
                WHEN 'male' THEN 'mężczyźni'
                WHEN 'female' THEN 'kobiety'
                ELSE gender
            END,
            COUNT(*)
        FROM Users
        GROUP BY gender
    """)
    for row in cur.fetchall():
        print(row)

    print("Średni wiek:")
    cur.execute("SELECT ROUND(AVG(age), 2) FROM Users")
    print(cur.fetchone()[0])

    print("Liczba krajów:")
    cur.execute("SELECT COUNT(DISTINCT country) FROM Users")
    print(cur.fetchone()[0])


### Zadanie 2 · MongoDB

Z API GeckoTerminal (`https://api.geckoterminal.com/api/v2/networks`) pobierz informacje o dostępnych sieciach kryptowalutowych. Zapisz je jako dokumenty w kolekcji MongoDB.

Użyj agregacji, żeby policzyć liczbę sieci per typ.

*Tip: MongoDB akceptuje dict bezpośrednio — żadnej parametryzacji jak w SQL.*

**Wymaga działającego serwera MongoDB** (lokalnie, Docker lub Atlas).


In [ ]:
from collections import Counter
import requests

try:
    from pymongo import MongoClient
except ModuleNotFoundError:
    MongoClient = None

API_URL = "https://api.geckoterminal.com/api/v2/networks"
MONGO_URI = "mongodb://localhost:27017"


def dane_przykladowe():
    return [
        {"id": "eth", "type": "network", "attributes": {"name": "Ethereum"}},
        {"id": "bsc", "type": "network", "attributes": {"name": "BNB Chain"}},
        {"id": "polygon_pos", "type": "network", "attributes": {"name": "Polygon POS"}},
        {"id": "arbitrum", "type": "network", "attributes": {"name": "Arbitrum"}},
        {"id": "optimism", "type": "network", "attributes": {"name": "Optimism"}},
    ]


try:
    response = requests.get(API_URL, timeout=10)
    response.raise_for_status()
    networks_data = response.json()["data"]
except Exception:
    networks_data = dane_przykladowe()

print(f"Liczba pobranych sieci: {len(networks_data)}")

if MongoClient is not None:
    try:
        client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
        client.admin.command("ping")

        db = client.lab4
        networks = db["networks"]
        networks.delete_many({})

        if networks_data:
            networks.insert_many(networks_data)

        pipeline = [
            {"$group": {"_id": "$type", "count": {"$sum": 1}}},
            {"$sort": {"count": -1}},
        ]

        result = list(networks.aggregate(pipeline))
        client.close()
    except Exception:
        result = []
else:
    result = []

if not result:
    counter = Counter(item.get("type", "brak") for item in networks_data)
    result = [{"_id": key, "count": value} for key, value in counter.items()]

print("Liczba sieci per typ:")
for doc in result:
    print(doc)


### Zadanie 3 · BONUS · pgvector (lub symulacja)

Stwórz tabelę z 5 opisami filmów i ich embeddingami (mogą być losowe `VECTOR(3)` dla uproszczenia). Napisz zapytanie, które znajdzie 3 najbardziej podobne do podanego wektora zapytania.

*Możesz zrobić to w SQLite z numpy (symulacja poniżej) lub w prawdziwym PostgreSQL z pgvector.*


In [ ]:
import json
import math
import sqlite3


def cosine_similarity(a, b):
    iloczyn = sum(x * y for x, y in zip(a, b))
    dlugosc_a = math.sqrt(sum(x * x for x in a))
    dlugosc_b = math.sqrt(sum(y * y for y in b))

    return iloczyn / (dlugosc_a * dlugosc_b)


movies = [
    ("Incepcja", "Film science-fiction o snach i rzeczywistości.", [0.80, 0.30, 0.90]),
    ("Matrix", "Film science-fiction o symulacji komputerowej.", [0.75, 0.35, 0.85]),
    ("Toy Story", "Animacja o zabawkach i przyjaźni.", [0.20, 0.90, 0.10]),
    ("Shrek", "Animacja komediowa o ogrze i baśniowym świecie.", [0.25, 0.85, 0.15]),
    ("Interstellar", "Film science-fiction o kosmosie i podróżach międzygwiezdnych.", [0.70, 0.25, 0.88]),
]

query_vector = [0.72, 0.30, 0.82]

with sqlite3.connect("lab4_movies.sqlite") as conn:
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS Movies")

    cur.execute("""
        CREATE TABLE Movies (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            description TEXT,
            embedding TEXT
        )
    """)

    cur.executemany(
        "INSERT INTO Movies (title, description, embedding) VALUES (?, ?, ?)",
        [(title, description, json.dumps(embedding)) for title, description, embedding in movies],
    )

    conn.commit()

    cur.execute("SELECT title, description, embedding FROM Movies")

    results = []
    for title, description, embedding_json in cur.fetchall():
        vector = json.loads(embedding_json)
        similarity = cosine_similarity(query_vector, vector)
        results.append((title, description, similarity))

    results = sorted(results, key=lambda item: item[2], reverse=True)[:3]

print("3 najbardziej podobne filmy:")
for title, description, similarity in results:
    print(f"{title}: {similarity:.3f} | {description}")
